# Mean and St.Dev. 

In [ ]:
import pandas as pd
import numpy as np
import re

# Leggi il file CSV
df = pd.read_csv('/Users/edoardoconti/Downloads/results_ordinal_paper/results_pretrained.csv')

# Estrai il tipo di esperimento da 'experiment'
df['experiment_type'] = df['experiment'].apply(lambda x: re.search(r'_(obd|clm|resnet18|resnetQWK|cnnregstn)', x).group(1) if re.search(r'_(obd|clm|resnet18|resnetQWK|cnnregstn)', x) else 'unknown')

# rename
df['experiment_type'] = df['experiment_type'].replace('resnet18', 'resnet18 CCE')
df['experiment_type'] = df['experiment_type'].replace('resnetQWK', 'resnet18 QWK')

# Definisci le metriche da calcolare
metrics = ["ccr", "f1", "acc_1off", "acc_2off", "qwk", "spearman", "ms", "mae", "rmse"]

# Funzione per calcolare media e deviazione standard delle metriche
def calculate_stats(experiment_df):
    mean_values = experiment_df[metrics].mean()
    std_values = experiment_df[metrics].std()
    return mean_values, std_values

# Calcola media e deviazione standard per ogni tipo di esperimento
experiment_types = df['experiment_type'].unique()

# Stampa le medie e le deviazioni standard per ogni tipo di esperimento
for exp_type in experiment_types:
    exp_df = df[df['experiment_type'] == exp_type]
    mean_values, std_values = calculate_stats(exp_df)
    
    print(f"\nExperiment Type: {exp_type}")
    print("Mean Values:")
    print(mean_values)
    print("\nStandard Deviation Values:")
    print(std_values)

# Boxplot

In [ ]:
import os
import matplotlib.pyplot as plt

# Percorso della cartella in cui salvare i boxplot
save_folder = '/Users/edoardoconti/Downloads/results_ordinal_paper/materiale_overleaf/boxplots'  
os.makedirs(save_folder, exist_ok=True)

for metric in metrics:
    plt.figure(figsize=(8, 4)) 
    positions = np.arange(len(experiment_types))
    positions = positions[::-1] #reverse

    medianprops = dict(linestyle='-', color='black')
    
    # Creazione del boxplot con colori
    box_plot = plt.boxplot([df[df['experiment_type'] == exp_type][metric] for exp_type in experiment_types], 
                           positions=positions, 
                           labels=[exp_type.upper() for exp_type in experiment_types],  # Rende maiuscole le etichette
                           showfliers=False,
                           vert=False,
                           widths=0.65,
                           patch_artist=True,
                           medianprops=medianprops
                           )
    
    # Specifica i colori e le gradazioni
    #colors = plt.cm.YlGn(np.linspace(0.1, 0.5, len(experiment_types)))
    colors = plt.cm.Oranges(np.linspace(0.1, 0.5, len(experiment_types)))
    
    # Assegna colori ai boxplot
    for i, box in enumerate(box_plot['boxes']):
        box.set_facecolor(colors[i])
    
    # Modifica la dimensione del font per le etichette
    plt.xlabel(metric.upper(), fontsize=20, labelpad=15)  # Aggiunge spazio sotto l'etichetta dell'asse x
    
    # Modifica la dimensione del font per le etichette degli assi
    plt.tick_params(axis='both', labelsize=15)
    
    # plt.show()

    # Costruisci il nome del file e salva l'immagine
    file_name = f'{save_folder}/{metric.upper()}.png'
    plt.savefig(file_name, bbox_inches='tight') 
    plt.close() 

    print(f'Salvato: {file_name}')

# Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(cm, title='Confusion matrix', cmap='Oranges', cbar=False):
    plt.figure(figsize=(8, 6)) if cbar else plt.figure(figsize=(6.2, 6))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap=cmap, cbar=cbar, annot_kws={"size": 18})
    plt.title(title, size=20, pad=15)
    plt.xlabel('Predicted', size=14)
    plt.ylabel('True', size=14)
    plt.xticks(np.arange(cm.shape[1]) + 0.5, range(cm.shape[1]), fontsize=14)
    plt.yticks(np.arange(cm.shape[0]) + 0.5, range(cm.shape[0]), fontsize=14, rotation=0)
    plt.show()

def plot_multiple_confusion_matrices(cms):
    for title, cm in cms:
        plot_confusion_matrix(cm, title=title)


# results from specific fold and split
cm_cnnregstn_sord = ["CNNRegSTN (SORD)", np.array([[0.72, 0.28, 0.00, 0.00],
                                             [0.23, 0.58, 0.19, 0.00],
                                             [0.14, 0.18, 0.56, 0.12],
                                             [0.13, 0.00, 0.17, 0.70]])]

cm_resnet_cce = ["ResNet18 (CCE)", np.array([[0.50, 0.39, 0.11, 0.00],
                                             [0.29, 0.41, 0.30, 0.00],
                                             [0.15, 0.19, 0.65, 0.01],
                                             [0.02, 0.02, 0.52, 0.44]])]

cm_resnet_qwk = ["ResNet18 (QWK)", np.array([[0.52, 0.45, 0.03, 0.00],
                                             [0.28, 0.53, 0.19, 0.00],
                                             [0.11, 0.27, 0.61, 0.01],
                                             [0.01, 0.02, 0.35, 0.62]])]

cm_obd_mse = ["OBD (MSE)", np.array([[0.64, 0.30, 0.06, 0.00],
                                     [0.23, 0.47, 0.30, 0.00],
                                     [0.14, 0.16, 0.69, 0.01],
                                     [0.02, 0.01, 0.31, 0.66]])]

cm_clm_qwk = ["CLM (QWK)", np.array([[0.54, 0.44, 0.02, 0.00],
                                     [0.24, 0.59, 0.17, 0.00],
                                     [0.08, 0.31, 0.60, 0.01],
                                     [0.00, 0.01, 0.32, 0.67]])]

# Lista delle matrici di confusione con i rispettivi titoli
cms = [cm_cnnregstn_sord, cm_resnet_cce, cm_resnet_qwk, cm_obd_mse, cm_clm_qwk]

# Chiamata alla funzione per stampare tutte le matrici
plot_multiple_confusion_matrices(cms)

### F1-score check test

In [ ]:
import numpy as np

# Estrazione della matrice di confusione
_, cm = cm_cnnregstn_sord

# Calcolo delle metriche
true_positives = np.diag(cm)
false_positives = cm.sum(axis=0) - true_positives
false_negatives = cm.sum(axis=1) - true_positives
true_negatives = cm.sum() - (true_positives + false_positives + false_negatives)

# Calcolo della precisione e richiamo
precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)

# Calcolo dell'F1 Score
f1_scores = 2 * (precision * recall) / (precision + recall)
f1_weighted = np.sum(f1_scores * (true_positives + false_negatives)) / np.sum(cm)

print(f"F1 Score: {f1_weighted:.2f}")

# Statistical analysis

In [ ]:
%pip install scikit-posthocs --quiet

In [ ]:
import pandas as pd
import re

# Funzione per estrarre il nome del modello usando regex
def extract_model(experiment):
    match = re.search(r'_(cnnregstn|resnet18|resnetQWK|obd|clm)_', experiment)
    return match.group(1) if match else None

# Funzione per estrarre fold e split
def extract_fold_split(experiment):
    match = re.search(r'fold(\d+)_split(\d+)', experiment)
    return (int(match.group(1)), int(match.group(2))) if match else (None, None)

def rank_ascending(metric):
    return metric in {'mae', 'rmse'}

# parametri 
file_path = '/Users/edoardoconti/Downloads/results_ordinal_paper/results_pretrained.csv' # csv con i risultati
alpha = 0.05
metric = 'qwk'

# lettura csv risultati
df = pd.read_csv(file_path)

# Applicare la funzione di estrazione del modello alla colonna "experiment"
df['measures'] = df['experiment'].apply(extract_model)

# Applicare la funzione di estrazione di fold e split alla colonna "experiment"
df[['fold', 'split']] = df['experiment'].apply(lambda x: pd.Series(extract_fold_split(x)))

# Creare un DataFrame pivot con i modelli come colonne, fold e split come indice, e QWK come valori
df_measures = df.pivot_table(index=['fold', 'split'], columns='measures', values=metric)

# Rimuovere le colonne fold e split
df_measures.reset_index(drop=True, inplace=True)

# Mostrare il DataFrame risultante
df_measures.head()

In [ ]:
# Creare la ranking table
df_rank = df_measures.rank(axis=1, method='min', ascending=rank_ascending(metric))

# Aggiungere una riga con la media dei rank per ciascun modello
df_rank.loc['average'] = df_rank.mean()

# Mostrare la ranking table risultante
#df_rank.tail()
df_rank

In [ ]:
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp

# Convertire il DataFrame in array per il test di Friedman
array_measures = [df_measures[col].values for col in df_measures.columns]

# settare valore di alpha
alpha = 0.05

# Test di Friedman
stat, p_value = friedmanchisquare(*array_measures)

print(f"\033[1mSettings:\033[0m")
print(f"alpha = {alpha}")
print(f"metric evaluated = {metric}\n")

print(f"\033[1mFriedman test results:\033[0m")
print(f"Statistic = {stat}")
print(f"p-value = {p_value}\n")

# Test di Nemenyi (se p-value del Friedman test è significativo)
print(f"\033[1mNemenyi post-hoc test results:\033[0m")
if p_value < alpha:
    # scikit_posthocs nemenyi test
    nemenyi_results = sp.posthoc_nemenyi_friedman(df_measures.values)
    print(f"{nemenyi_results}\n")
    
    # individuare dalla tabella del test post hoc nemenyi se ci sono confronti significativi
    significant_results = [
        (df_measures.columns[i], df_measures.columns[j], nemenyi_results.iloc[i, j],
        df_rank.columns[i] if df_rank.loc['average', df_measures.columns[i]] < df_rank.loc['average', df_measures.columns[j]] else df_rank.columns[j])
        for i in range(nemenyi_results.shape[0]) 
        for j in range(i + 1, nemenyi_results.shape[1]) 
        if nemenyi_results.iloc[i, j] < alpha
    ]
    
    if significant_results:
        for model1, model2, p_value, better_model in significant_results:
            print(f"Significant comparison between {model1.upper()} e {model2.upper()} with p-value = {p_value:.5f} (best model: {better_model.upper()}).")
    else:
        print(f"No significant comparison between models.")
else:
    print("There are no significant differences between the groups according to the Friedman test.")

### Manual computation of tests

In [ ]:
from scipy.stats import f

# Calcolare Q
n = len(df_rank) - 1  # il numero di righe è 15
k = len(df_rank.columns)  # il numero di modelli è 5
mean_ranks = df_rank.loc['average'].values

# chi-square
Q = (12 * n) / (k * (k + 1)) * np.sum((mean_ranks - (k + 1) / 2) ** 2)
# chi_squared_F = (12 * n) / (k * (k + 1)) * (np.sum(mean_ranks ** 2) - (k * (k + 1) ** 2) / 4) # formulazione alternativa a Q
FF = ((n - 1) * Q) / ((n * (k - 1)) - Q)

# Definire i gradi di libertà
dfn = k - 1  # Gradi di libertà del numeratore
dfd = (k-1) * (n-1)  # Gradi di libertà del denominatore

# Livello di significatività
alpha = 0.05

# Calcolare il valore critico F
F_critical = f.ppf(1 - alpha, dfn, dfd)

print(f"chi-square_F = {Q}")
print(f"F_F = {FF}\n")
print(f"Critic value F: {F_critical}\n")
print(f"Statistically significant: {FF > F_critical}")

In [ ]:
import numpy as np

q_alpha = 2.728
CD = q_alpha * np.sqrt((k * (k + 1)) / (6 * n))

# Mostrare il valore critico per il test di Nemenyi
print(f'Critical value for Nemenyi test: {CD:.4f}\n')

# Estrarre le mean_rank
mean_ranks = df_rank.loc['average'].values

# Calcolare tutte le differenze tra ogni coppia di mean_rank
differences = []
for i in range(k):
    for j in range(i + 1, k):
        diff = abs(mean_ranks[i] - mean_ranks[j])
        differences.append((df_rank.columns[i], df_rank.columns[j], diff, diff > CD))

# Mostrare le differenze e il confronto con il valore critico
for model1, model2, diff, significant in differences:
    result = "Significant" if significant else "Not significant"
    print(f'{model1} vs {model2}: {diff:.4f} ({result})')